In [1]:
import requests
from urllib.parse import quote_plus
from pydantic_ai import Agent, RunUsage

In [2]:
instructions = """
You are a Wikipedia assistant. Your task is to answer user questions using Wikipedia as your source.
Step 1 – Identify the topic. Determine the central subject of the user's question.
Step 2 – Fetch or search.

If the user provides a Wikipedia URL, decode the title from the URL and call get_page directly.
If the user names a specific Wikipedia article, call get_page directly.
Otherwise, call search_wikipedia with the central topic. From the results, select the 1–2 most relevant pages and fetch them with get_page.

Step 3 – Handle edge cases.

If a page is a disambiguation page, identify the most relevant entry for the user's question, or ask the user to clarify.
If no results are found, inform the user and suggest a rephrased query.

Step 4 – Respond. Answer the user's question in 2–4 sentences using only information from the fetched page(s). If the page doesn't fully answer the question, say so.
"""

In [3]:
agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions
)

In [4]:
@agent.tool_plain
def search_wikipedia(query: str) -> list[dict]:
    """
    Searches wikipedia for articles matching the query

    Args:
        query (str): The search query to look up on Wikipedia.

    Returns:
        list: A list of search result objects from wikipedia API
    """
    query = quote_plus(query)
    url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
    response = requests.get(url, headers=headers)
    res = response.json()
    searches = res.get("query", {}).get("search", [])
    return searches

@agent.tool_plain
def get_page(page_title: str) -> str:
    """
    Fetch the content of the article matching the page title

    Args:
        page_title (str): The title of the wikipedia page

    Returns:
        str: The content of the wikipedia page
    """
    page_title = quote_plus(page_title)
    url = f"https://en.wikipedia.org/w/index.php?title={page_title}&action=raw"
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
    response = requests.get(url, headers=headers)
    return response.text

In [6]:
## Question 1

res = search_wikipedia("capybara")
len(res)

10

In [12]:
## Question 2

count = sum(1 for item in res if "capybara" in item.get('title').lower())
count

5

In [16]:
## Question 3

page = get_page("Capybara")
len(page)

36946

In [17]:
## Question 4 - PydanticAI + OpenAI

In [18]:
## Question 5

query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

result = await agent.run(query)

In [20]:
## inspecting messages

def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [25]:
output = result.output

messages = result.all_messages()

In [26]:
print_messages(messages)

request
USER: What is this page about? https://en.wikipedia.org/wiki/Capybara

response
TOOL CALL: get_page {"page_title":"Capybara"}

request
TOOL RETURN: get_page

response
The Wikipedia page on **Capybara** provides detailed information about this species, which is the largest living rodent, found throughout South America excluding Chile. Known scientifically as *Hydrochoerus hydrochaeris*, the capybara is a semiaquatic herbivore that typically inhabits savannas and dense forests near water bodies. They are highly social animals, often living in groups that can number from 10 to over 100 individuals. The page also covers their diet, physical characteristics, behavior, reproduction, and conservation status, noting that they are not considered threatened overall despite some hunting pressures and habitat loss. Additionally, the capybara has gained popularity in various cultures and on social media due to their docile nature.



In [27]:
## Question 6
query = "What are the main threats to capybara populations?"
result = await agent.run(query)
messages = result.all_messages()

In [28]:
print_messages(messages)

request
USER: What are the main threats to capybara populations?

response
TOOL CALL: search_wikipedia {"query":"Capybara threats"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: search_wikipedia {"query":"Capybara"}

request
TOOL RETURN: search_wikipedia

response
TOOL CALL: get_page {"page_title":"Capybara"}

request
TOOL RETURN: get_page

response
The main threats to capybara populations include hunting and habitat loss. Capybaras are hunted for their meat and pelts in various regions, which has led to reduced numbers in some areas. Additionally, they often face competition with livestock for grazing resources, resulting in conflicts with humans. Despite these threats, capybaras are currently not considered endangered, and their populations remain stable overall in many parts of South America, thanks in part to their ability to breed rapidly and their adaptability to urban environments.



In [ ]:
# loop

messages = []
usage = RunUsage()



while True:
    user_prompt = input('Search:')
    if user_prompt.lower().strip() == 'stop':
        break

    result = await agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage()

    print_messages(result.new_messages())
    messages.extend(result.new_messages())